In [4]:
# Cellule 1 — Charger les librairies
import os
import cv2
from ultralytics import YOLO

print("Librairies chargées !")

Librairies chargées !


In [7]:
# Cellule 2 — Configurer les dossiers
import os

# Se placer dans le bon dossier
os.chdir("C:/Users/PCµ/Documents/Cours-VIT-/projet_VIT_NLP")

os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/lines", exist_ok=True)
os.makedirs("data/bilevel", exist_ok=True)

images = sorted(os.listdir("data/images"))
print(f"{len(images)} images trouvées")
print(f"Dossier actuel : {os.getcwd()}")

34 images trouvées
Dossier actuel : C:\Users\PCµ\Documents\Cours-VIT-\projet_VIT_NLP


In [3]:
# Cellule 3 — Détecter les lignes avec YOLO sur une page test
model = YOLO("yolov8n.pt")  # charger le modèle YOLO (téléchargement automatique)

# Tester sur la première page
img = cv2.imread("data/processed/page_001.jpg")
results = model(img)

# Compter les zones détectées
boxes = results[0].boxes.xyxy
print(f"{len(boxes)} zones détectées sur la page 1")


0: 640x448 (no detections), 246.7ms
Speed: 12.7ms preprocess, 246.7ms inference, 10.4ms postprocess per image at shape (1, 3, 640, 448)
0 zones détectées sur la page 1


In [4]:
# Cellule 4 — Segmentation et transcription avec Kraken
import subprocess

# Chemin du modèle téléchargé
model_path = r"C:\Users\PCµ\AppData\Local\htrmopo\htrmopo\96b0fb58-0a8d-5e49-896f-a0ea59f36d62\HTR-United-Manu_McFrench.mlmodel"

# Tester sur la première page
result = subprocess.run([
    "kraken", "-i", "data/processed/page_001.jpg",
    "data/lines/page_001.txt",
    "segment", "ocr", "-m", model_path
], capture_output=True, text=True, env={**os.environ, "PYTHONUTF8": "1"})

print(result.stdout)
print(result.stderr)

Segmenting data\processed\page_001.jpg	[06/18/26 16:04:50] ERROR    Image data\processed\page_001.jpg   pageseg.py:356
                             is not bi-level                                   
âœ—
                    ERROR    Failed processing                    kraken.py:424
                             data\processed\page_001.jpg: 1                    




In [8]:
# Cellule 5 — Convertir les images en bi-level pour Kraken
from PIL import Image
import numpy as np

os.makedirs("data/bilevel", exist_ok=True)

for filename in images:
    # Charger l'image traitée
    img = cv2.imread(f"data/processed/{filename}", cv2.IMREAD_GRAYSCALE)
    # Convertir en vraiment noir/blanc (bi-level)
    _, bilevel = cv2.threshold(img, 128, 255, cv2.THRESH_BINARY)
    # Sauvegarder en PNG (meilleur format pour Kraken)
    outname = filename.replace(".jpg", ".png")
    cv2.imwrite(f"data/bilevel/{outname}", bilevel)

print(f"{len(os.listdir('data/bilevel'))} images bi-level créées")

34 images bi-level créées


In [6]:
# Cellule 6 — Transcription avec Kraken sur image bi-level
result = subprocess.run([
    "kraken", "-i", "data/bilevel/page_001.png",
    "data/lines/page_001.txt",
    "segment", "ocr", "-m", model_path
], capture_output=True, text=True, env={**os.environ, "PYTHONUTF8": "1"})

print(result.stdout)
print(result.stderr)

Segmenting data\bilevel\page_001.png	âœ“

Processing ----------------------------------------   0% 0/21 -:--:-- 0:00:00
Processing ----------------------------------------   0% 0/21 -:--:-- 0:00:01
Processing ----------------------------------------   0% 0/21 -:--:-- 0:00:02
Processing ----------------------------------------   0% 0/21 -:--:-- 0:00:03
Processing ----------------------------------------   0% 0/21 -:--:-- 0:00:04
Processing ----------------------------------------   0% 0/21 -:--:-- 0:00:05
Processing ----------------------------------------   0% 0/21 -:--:-- 0:00:06
Processing ----------------------------------------   0% 0/21 -:--:-- 0:00:07
Processing ----------------------------------------   0% 0/21 -:--:-- 0:00:08
Processing ----------------------------------------   0% 0/21 -:--:-- 0:00:09
Processing ----------------------------------------   0% 0/21 -:--:-- 0:00:10
Processing ----------------------------------------   0% 0/21 -:--:-- 0:00:11
Processing -----------

In [7]:
# Cellule 7 — Lire le texte transcrit
with open("data/lines/page_001.txt", "r", encoding="utf-8") as f:
    texte = f.read()

print("=== TEXTE TRANSCRIT ===")
print(texte)

=== TEXTE TRANSCRIT ===
(nt. (Callice
14s
DO kOC ANA
(CS lOttrs CnvonOCs AuTOy
M De Ma7ypOC A
MAOOG
Nostvosivo anOssCianCurs do
M Nu n 111
Merali N441
M 4 1°v aau4 13 i1
wwI Ao BmOo
I 
AaUIUIOIIE, uS UUIIIIES oi 
M° II NIn4 M Mun4 N11
M NI non
Mlanokol n M
Sol dolavile do Paris,
M 1uusourU s
1.
1000
Source gallica. Ont. tr (Bibliotheque nationdle de France


In [2]:
# Vérification des fichiers bilevel
import os
os.chdir("C:/Users/PCµ/Documents/Cours-VIT-/projet_VIT_NLP")

pages_a_verifier = [
    "page_003.png", "page_013.png", "page_014.png", "page_015.png",
    "page_016.png", "page_017.png", "page_018.png",
    "page_019.png", "page_020.png"
]

for p in pages_a_verifier:
    chemin = f"data/bilevel/{p}"
    statut = "✓ OK" if os.path.exists(chemin) else "✗ MANQUANT"
    print(f"{statut} — {chemin}")

✓ OK — data/bilevel/page_003.png
✓ OK — data/bilevel/page_013.png
✓ OK — data/bilevel/page_014.png
✓ OK — data/bilevel/page_015.png
✓ OK — data/bilevel/page_016.png
✓ OK — data/bilevel/page_017.png
✓ OK — data/bilevel/page_018.png
✓ OK — data/bilevel/page_019.png
✓ OK — data/bilevel/page_020.png


In [ ]:
# Cellule 8 — Transcrire uniquement page_014
import subprocess
import os

os.chdir("C:/Users/PCµ/Documents/Cours-VIT-/projet_VIT_NLP")

model_path = r"C:\Users\PCµ\AppData\Local\htrmopo\htrmopo\96b0fb58-0a8d-5e49-896f-a0ea59f36d62\HTR-United-Manu_McFrench.mlmodel"

result = subprocess.run([
    "kraken", "-i", "data/bilevel/page_014.png",
    "data/lines/page_014.txt",
    "segment", "ocr", "-m", model_path
], capture_output=True, text=True,
   env={**os.environ, "PYTHONUTF8": "1"})